# Approach C: Forecast + Classify (Recommended)

**The hybrid approach:**
1. Use **Chronos-2** (pretrained time series foundation model) deployed on a **SageMaker AI real-time endpoint** to forecast expected consumption
2. Compute **residuals** (actual - forecast) — this absorbs seasonality and trend
3. Leverage **full probabilistic output** (quantiles 0.1–0.9) for anomaly detection features
4. Engineer features on the **residuals** with temporal structure, holiday awareness, and weather correlation
5. Train XGBoost classifier on the enriched residual + quantile features

> **Why this works:** A thief's actual consumption falls *below* the forecast prediction interval. By using all 9 Chronos quantiles, we detect when actual < q0.1 (10th percentile) — a strong theft signal that Approach B cannot compute. We also apply the same temporal feature vocabulary as B (weekday/weekend, periodicity, rolling volatility, weather correlation) but on *residuals*, where seasonality has already been removed.

> **Chronos-2 is deployed once as a SageMaker endpoint — no per-meter training, no retraining pipeline.**
>
> | Component | Needs Training? | SageMaker Resource | Retraining Cadence |
> |-----------|----------------|-------------------|-------------------|
> | **Chronos-2** (forecasting) | **No** — pretrained, zero-shot | **Real-time endpoint** (JumpStart) | **Never** |
> | **XGBoost** (classification) | Yes — on residual features | Training job (`ModelTrainer`) | Quarterly |

**Prerequisite:** Deploy a Chronos-2 endpoint from JumpStart before running this notebook.

**Infrastructure:** Chronos-2 on SageMaker endpoint (GPU). XGBoost trained on SageMaker via SDK v3.

In [ ]:
import json
import warnings
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support

from utils import (
    get_sagemaker_session, load_sgcc_dataset, preprocess,
    compute_residual_features, upload_to_s3,
    train_xgboost_on_sagemaker, download_and_load_model, save_results,
    ROLE, PREFIX,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Imports OK")

In [ ]:
# SageMaker session
sagemaker_session, region, bucket = get_sagemaker_session()
s3_client = boto3.client("s3")
sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=region)
print(f"Region: {region} | Bucket: {bucket}")

In [ ]:
# Load and preprocess SGCC dataset
raw_df = load_sgcc_dataset()
customer_ids, labels, consumption_clean, date_columns, train_idx, test_idx = preprocess(raw_df)

## Deploy Chronos-2 Endpoint (run once)

Uncomment and run the cell below to deploy Chronos-2 from JumpStart. Skip if already deployed.

In [ ]:
# Uncomment to deploy Chronos-2 endpoint from JumpStart
#
# from sagemaker.jumpstart.model import JumpStartModel
#
# chronos_model = JumpStartModel(
#     model_id="amazon-chronos-2-base",  # or amazon-chronos-2-large
#     role=ROLE,
#     sagemaker_session=sagemaker_session,
# )
# predictor = chronos_model.deploy(
#     initial_instance_count=1,
#     instance_type="ml.g5.xlarge",
#     endpoint_name="chronos-2-ntl-demo",
# )
# print("Endpoint deployed: chronos-2-ntl-demo")

CHRONOS_ENDPOINT_NAME = "chronos-2-ntl-demo"
print(f"Using endpoint: {CHRONOS_ENDPOINT_NAME}")

## Chronos-2 Inference: Generate Forecasts & Compute Residuals

In [ ]:
def invoke_chronos_endpoint(context_series, prediction_length=64):
    """Invoke Chronos-2 SageMaker endpoint for batch forecasting.

    Args:
        context_series: np.ndarray of shape (n_customers, context_length)
        prediction_length: number of steps to forecast

    Returns:
        dict with keys:
            'median': np.ndarray (n_customers, prediction_length) — p50 forecasts
            'q01': np.ndarray (n_customers, prediction_length) — 10th percentile
            'q09': np.ndarray (n_customers, prediction_length) — 90th percentile
    """
    all_median, all_q01, all_q09 = [], [], []
    batch_size = 64

    for batch_start in range(0, len(context_series), batch_size):
        batch_end = min(batch_start + batch_size, len(context_series))
        batch = context_series[batch_start:batch_end]

        payload = {
            "inputs": [
                {"target": row.tolist()}
                for row in batch
            ],
            "parameters": {
                "prediction_length": prediction_length,
                "quantile_levels": [0.1, 0.5, 0.9],
            },
        }

        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=CHRONOS_ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps(payload),
        )
        result = json.loads(response["Body"].read().decode())

        for pred in result["predictions"]:
            all_median.append(pred["0.5"])
            all_q01.append(pred["0.1"])
            all_q09.append(pred["0.9"])

    return {
        "median": np.array(all_median),
        "q01": np.array(all_q01),
        "q09": np.array(all_q09),
    }

print("Endpoint invocation function defined (returning q0.1, q0.5, q0.9).")

In [ ]:
# Rolling forecast: generate residuals and quantile bounds across the evaluation window
CONTEXT_LEN = 512
PRED_LEN = 64

cons_values = consumption_clean.values
n_customers = cons_values.shape[0]

n_rolls = (cons_values.shape[1] - CONTEXT_LEN) // PRED_LEN
print(f"Rolling forecast: {n_rolls} rolls of {PRED_LEN} days each")
print(f"Total residual coverage: {n_rolls * PRED_LEN} days")

all_residuals = np.full((n_customers, n_rolls * PRED_LEN), np.nan)
all_actuals = np.full_like(all_residuals, np.nan)
all_q01 = np.full_like(all_residuals, np.nan)
all_q09 = np.full_like(all_residuals, np.nan)

for roll in range(n_rolls):
    ctx_start = roll * PRED_LEN
    ctx_end = ctx_start + CONTEXT_LEN
    pred_start = ctx_end
    pred_end = pred_start + PRED_LEN

    if pred_end > cons_values.shape[1]:
        break

    print(f"  Roll {roll + 1}/{n_rolls}: context[{ctx_start}:{ctx_end}] -> forecast[{pred_start}:{pred_end}]", end="")

    context_batch = np.nan_to_num(cons_values[:, ctx_start:ctx_end], nan=0.0)
    forecast_result = invoke_chronos_endpoint(context_batch, prediction_length=PRED_LEN)

    actuals = cons_values[:, pred_start:pred_end]
    residuals = actuals - forecast_result["median"]

    col_start = roll * PRED_LEN
    col_end = col_start + PRED_LEN
    all_residuals[:, col_start:col_end] = residuals
    all_actuals[:, col_start:col_end] = actuals
    all_q01[:, col_start:col_end] = forecast_result["q01"]
    all_q09[:, col_start:col_end] = forecast_result["q09"]

    print(f" | mean|residual|={np.nanmean(np.abs(residuals)):.2f}")

print(f"\nResidual matrix shape: {all_residuals.shape}")
print(f"Coverage: {(~np.isnan(all_residuals)).mean():.1%}")

In [ ]:
# Visualize: Normal vs Theft residual patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

np.random.seed(42)
normal_idx = np.random.choice(labels[labels == 0].index, 3, replace=False)
theft_idx = np.random.choice(labels[labels == 1].index, 3, replace=False)

ax = axes[0]
for idx in normal_idx:
    ax.plot(all_residuals[idx], alpha=0.7, linewidth=0.5)
ax.axhline(y=0, color="black", linestyle="--", linewidth=0.5)
ax.set_title("Residuals: Normal Customers")
ax.set_ylabel("Actual - Forecast (kWh)")
ax.set_xlabel("Day")

ax = axes[1]
for idx in theft_idx:
    ax.plot(all_residuals[idx], alpha=0.7, linewidth=0.5, color="red")
ax.axhline(y=0, color="black", linestyle="--", linewidth=0.5)
ax.set_title("Residuals: Theft Customers")
ax.set_ylabel("Actual - Forecast (kWh)")
ax.set_xlabel("Day")

plt.suptitle("Forecast Residuals — The Signal the Classifier Actually Sees", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Residual Feature Engineering & Training

In [ ]:
# Compute residual-based features (with quantile anomaly + temporal + holiday enrichment)
residual_dates = date_columns[CONTEXT_LEN : CONTEXT_LEN + all_residuals.shape[1]]

quantile_data = {
    "actuals": all_actuals,
    "q01": all_q01,
    "q09": all_q09,
}

print(f"Computing residual features with quantile, temporal, and holiday enrichment over {len(residual_dates)} days...")
features_c = compute_residual_features(all_residuals, dates=residual_dates, quantile_data=quantile_data)
features_c.index = consumption_clean.index
features_c["FLAG"] = labels.values
print(f"Feature set: {features_c.shape[1] - 1} features")
features_c.describe()

In [ ]:
# Split and upload to S3
train_c = features_c.iloc[train_idx]
test_c = features_c.iloc[test_idx]

train_s3 = upload_to_s3(train_c, f"{PREFIX}/approach-c/data/train/train.csv", bucket, s3_client)
test_s3 = upload_to_s3(test_c, f"{PREFIX}/approach-c/data/test/test.csv", bucket, s3_client)

In [ ]:
# Train on SageMaker
trainer_c = train_xgboost_on_sagemaker(
    "approach-c", train_s3, test_s3,
    region=region, bucket=bucket, sagemaker_session=sagemaker_session,
)

In [ ]:
# Load model and evaluate
model_c, metrics_c = download_and_load_model(trainer_c, s3_client)
X_test = test_c.drop(columns=["FLAG"])
y_test = test_c["FLAG"]

y_pred = model_c.predict(X_test)
y_prob = model_c.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("APPROACH C: FORECAST + CLASSIFY")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Normal", "Theft"]))

In [ ]:
# Save results for comparison notebook
save_results("approach-c", y_test, y_pred, y_prob, feature_names=list(X_test.columns))

## Feature Importance & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance
ax = axes[0]
importances = model_c.feature_importances_
feature_names = list(X_test.columns)
sorted_idx = np.argsort(importances)
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color="#2ecc71")
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance")
ax.set_title("Which Residual Features Detect Theft?")

# Confusion matrix
ax = axes[1]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=["Normal", "Theft"], yticklabels=["Normal", "Theft"], ax=ax)
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
ax.set_title("Approach C: Confusion Matrix")

plt.tight_layout()
plt.show()

print(f"False Positives: {cm[0,1]} (wasted inspections)")
print(f"True Positives:  {cm[1,1]} (caught theft)")
print(f"False Negatives: {cm[1,0]} (missed theft)")

## Cleanup

Uncomment to delete the Chronos-2 endpoint when done.

In [ ]:
# Uncomment to delete the Chronos-2 endpoint
# sagemaker_session.delete_endpoint(CHRONOS_ENDPOINT_NAME)
# print(f"Deleted endpoint: {CHRONOS_ENDPOINT_NAME}")